# ChurnLens — 03. Root Cause & Leading Indicators Analysis
## Separating Symptoms from True Root Causes & Ranking Early Warnings

### Core Analytical Thesis:
Instead of stopping at obvious symptoms (*"Customers who churn had fewer logins"*), we investigate:
**Why did their activity drop? What triggered the churn cascade?**

### Hypotheses Evaluated:
1. **Onboarding Failure**: Did early non-adoption of key features seal customer fate in months 1–3?
2. **Involuntary Billing Friction**: How much churn is driven purely by failed credit card payments?
3. **Support Escalation Breakdown**: Do unresolved tickets and formal complaints cause customer churn?
4. **Value & Pricing Cliff**: Do customers downgrade before cancelling?
5. **Leading Warning Signals**: Ranking behaviors that emerge 14–30 days BEFORE churn.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/cleaned_churn_data.csv')


### 1. Root Cause 1: Onboarding Quality & Key Feature Adoption


In [ ]:
feature_cross = pd.crosstab(
    df['key_feature_usage'], df['churned'], normalize='index'
) * 100
feature_cross.index = ['No Core Feature Adoption', 'Core Feature Adopted']
feature_cross.columns = ['Active (%)', 'Churned (%)']

print("Churn Rate by Core Feature Adoption:")
feature_cross


In [ ]:
# Interaction of Tenure and Feature Adoption on Churn Rate
onboarding_matrix = df.pivot_table(
    index='is_early_stage', columns='key_feature_usage', values='churned', aggfunc='mean'
) * 100
onboarding_matrix.index = ['Tenure > 3 Mo', 'Tenure 1-3 Mo (New)']
onboarding_matrix.columns = ['No Core Feature', 'Core Feature Adopted']

plt.figure(figsize=(7, 4.5))
sns.heatmap(onboarding_matrix, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={'label': 'Churn Rate (%)'})
plt.title('Churn Rate (%): Early Tenure x Feature Adoption Interaction', fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


### 2. Root Cause 2: Involuntary Billing & Payment Gateways


In [ ]:
billing_agg = df.groupby('failed_payments').agg(
    total_customers=('customer_id', 'count'),
    churn_rate=('churned', 'mean'),
    lost_mrr=('monthly_spend', lambda x: x[df.loc[x.index, 'churned'] == 1].sum())
).reset_index()

billing_agg['churn_rate_pct'] = (billing_agg['churn_rate'] * 100).round(2)
billing_agg


### 3. Root Cause 3: Support Friction & Escalation Resolution


In [ ]:
support_agg = df.groupby(['support_tickets', 'unresolved_tickets']).agg(
    customer_count=('customer_id', 'count'),
    churn_rate=('churned', 'mean')
).reset_index()

# Filter meaningful volume
support_pivot = support_agg[support_agg['customer_count'] >= 20].pivot(
    index='support_tickets', columns='unresolved_tickets', values='churn_rate'
) * 100

plt.figure(figsize=(8, 5))
sns.heatmap(support_pivot, annot=True, fmt=".1f", cmap="Reds", cbar_kws={'label': 'Churn Rate (%)'})
plt.title('Churn Rate (%): Total Support Tickets vs Unresolved Backlog', fontsize=11, fontweight='bold', pad=12)
plt.xlabel('Unresolved Tickets')
plt.ylabel('Total Support Tickets')
plt.tight_layout()
plt.show()


### 4. Root Cause 4: Plan Downgrades as Churn Precursors


In [ ]:
downgrade_summary = df.groupby('downgrades').agg(
    total_customers=('customer_id', 'count'),
    churned_customers=('churned', 'sum'),
    churn_rate=('churned', 'mean'),
    avg_mrr=('monthly_spend', 'mean')
).reset_index()
downgrade_summary['churn_rate_pct'] = (downgrade_summary['churn_rate'] * 100).round(2)
downgrade_summary


### 5. Primary Root Cause Attribution Breakdown


In [ ]:
churners_df = df[df['churned'] == 1]
cause_dist = churners_df['churn_reason_category'].value_counts().reset_index()
cause_dist.columns = ['Root Cause Category', 'Churn Count']
cause_dist['Percentage (%)'] = (cause_dist['Churn Count'] / len(churners_df) * 100).round(1)

plt.figure(figsize=(10, 5))
sns.barplot(data=cause_dist, y='Root Cause Category', x='Percentage (%)', palette='mako')
plt.title('Attributed Root Cause Distribution Among Churned Customers', fontsize=12, fontweight='bold', pad=12)
for i, v in enumerate(cause_dist['Percentage (%)']):
    plt.text(v + 0.5, i, f"{v}% ({cause_dist.loc[i, 'Churn Count']:,})", va='center', fontweight='bold')
plt.xlim(0, max(cause_dist['Percentage (%)']) + 12)
plt.tight_layout()
plt.show()


### 6. Leading Indicators Ranking Table


In [ ]:
indicators = [
    {'Signal': 'Inactivity >= 14 Days', 'Condition': df['days_since_last_login'] >= 14},
    {'Signal': 'Activity Velocity Drop > 40%', 'Condition': df['activity_change_pct'] <= -40},
    {'Signal': 'Failed Payment >= 1', 'Condition': df['failed_payments'] >= 1},
    {'Signal': 'Plan Downgrade Executed', 'Condition': df['downgrades'] > 0},
    {'Signal': 'Unresolved Support Ticket >= 1', 'Condition': df['unresolved_tickets'] >= 1},
    {'Signal': 'Core Feature Never Adopted', 'Condition': df['key_feature_usage'] == 0},
]

indicator_results = []
for ind in indicators:
    subset = df[ind['Condition']]
    total_flagged = len(subset)
    churners = subset['churned'].sum()
    churn_rate = (churners / total_flagged) * 100 if total_flagged > 0 else 0
    indicator_results.append({
        'Leading Warning Signal': ind['Signal'],
        'Total Customers Flagged': total_flagged,
        'Churned Count': churners,
        'Churn Probability When Flagged (%)': round(churn_rate, 2),
        'Hazard Multiplier vs Baseline': round(churn_rate / (df['churned'].mean() * 100), 2)
    })

indicator_df = pd.DataFrame(indicator_results).sort_values(by='Churn Probability When Flagged (%)', ascending=False)
indicator_df
